# 02. 미니 RLM 하네스 구현

목표: 컨텍스트 오프로딩과 프로그램식 서브 호출을 흉내 내는 작은 실행기를 구현합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 이 노트북은 실제 LLM을 호출하지 않고, 하네스 구조만 결정적으로 시뮬레이션합니다.

## 1. 데이터와 작업 명세 준비

루트 정책은 `RECORDS`와 `TASK`라는 변수 이름만 봅니다. 실제 레코드 내용과 도메인별 조건은 서브 함수가 처리합니다.

In [ ]:
def make_commerce_records(n=12):
    return [
        {"id": f"order-{i:03d}", "status": "late" if i % 4 == 0 else "ok", "amount": 100 + i * 3}
        for i in range(n)
    ]


def make_support_records(n=12):
    return [
        {"id": f"ticket-{i:03d}", "severity": "high" if i % 5 == 0 else "normal", "minutes": 15 + i * 2}
        for i in range(n)
    ]


commerce_task = {"filter_field": "status", "filter_value": "late", "sum_field": "amount"}
support_task = {"filter_field": "severity", "filter_value": "high", "sum_field": "minutes"}

commerce_records = make_commerce_records()
support_records = make_support_records()

print(commerce_records[:3])
print(support_records[:3])

## 2. 루트 정책과 서브 함수

루트 정책은 항상 같은 계획을 반환합니다. 이 점이 중요합니다. 루트는 긴 데이터와 도메인 단어를 보지 않고, 구조적인 실행 계획만 만듭니다.

In [ ]:
def root_policy(records_var="RECORDS", task_var="TASK"):
    # 루트 모델이 배워야 하는 것은 도메인 지식이 아니라 안정적인 분해 절차입니다.
    return [
        ("select_records", records_var, task_var, "SELECTED"),
        ("aggregate_records", "SELECTED", task_var, "SUMMARY"),
        ("format_answer", "SUMMARY", None, "ANSWER"),
    ]


def select_records(records, task):
    field = task["filter_field"]
    expected = task["filter_value"]
    return [record for record in records if record.get(field) == expected]


def aggregate_records(records, task):
    sum_field = task["sum_field"]
    return {
        "count": len(records),
        "sum_field": sum_field,
        "sum": sum(record.get(sum_field, 0) for record in records),
        "ids": [record["id"] for record in records],
    }


def format_answer(summary, _unused=None):
    return f"조건을 만족한 항목은 {summary['count']}개이고, {summary['sum_field']} 합계는 {summary['sum']}입니다."


SUBCALLS = {
    "select_records": select_records,
    "aggregate_records": aggregate_records,
    "format_answer": format_answer,
}

## 3. 하네스 실행기

실행기는 REPL처럼 변수 환경을 가지고 있습니다. 서브 호출 결과는 루트 프롬프트에 붙는 대신 변수에 저장됩니다.

In [ ]:
def run_harness(records, task):
    env = {"RECORDS": records, "TASK": task}
    root_trace = []

    for function_name, input_var, task_var, output_var in root_policy():
        root_trace.append(f"{function_name}({input_var}, {task_var}) -> {output_var}")
        function = SUBCALLS[function_name]
        input_value = env[input_var]
        task_value = env[task_var] if task_var else None
        env[output_var] = function(input_value, task_value)

    return env["ANSWER"], root_trace, env


commerce_answer, commerce_trace, commerce_env = run_harness(commerce_records, commerce_task)
support_answer, support_trace, support_env = run_harness(support_records, support_task)

print("commerce:", commerce_answer)
print("support: ", support_answer)
print("same root trace:", commerce_trace == support_trace)
print("root trace:")
for step in commerce_trace:
    print(" -", step)

## 4. 무엇을 배웠나

두 도메인의 데이터 필드와 값은 다르지만 루트 궤적은 완전히 같습니다. 원문이 말하는 trajectory isomorphism은 이런 구조를 더 큰 규모의 LLM 에이전트 학습에 적용하려는 아이디어입니다. 루트 정책은 동일한 분해 전략을 반복하고, 도메인별 세부 처리는 서브 호출로 내려갑니다.